# A1.12 · Resource overload

**Function A — Security Architecture & Platform → The Agentic Reference Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.11 · Cascading hallucination](https://spbreed.github.io/cyber-commons/lessons/A1.11.html)**.

| | |
|---|---|
| Open-source tooling | OpenTelemetry |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


**OWASP T4 — Resource Overload. LLM10 — Unbounded Consumption.**

The **agent_runtime** loops: plan, act, observe, decide again. The loop is the
component that makes an agent an agent, and a loop with no exit condition runs
until something outside it intervenes.

What intervenes, in practice, is a bill, a rate limit, or a person at 3am.

Four resources drain, and they fail differently:

**Tokens and money** — the visible one, discovered on an invoice.

**Downstream capacity** — the one that hurts other people. An agent retrying a
failing API in a tight loop is a denial-of-service attack on your own service,
launched from inside your perimeter by something with valid credentials.

**Rate limit budget** — shared with the humans who need it. The agent exhausts
the quota and the on-call engineer cannot query the API they need.

**Wall-clock time in a critical path** — a workflow step that never returns.

This is a security risk rather than a cost problem for two reasons. It is
**reachable by an attacker**: a task that cannot succeed is easy to construct
via A1.3, and costs the attacker nothing. And it is **availability**, which is
one third of the triad regardless of how the outage was caused.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```
\n## 2 · The risk, realised\n\nA task that cannot succeed, and a loop with no ceiling.

In [ ]:
DOWNSTREAM = {"calls": 0, "capacity": 50, "rejected": 0}

def flaky_api(query):
    """A downstream service. It is not broken - the query cannot be satisfied."""
    DOWNSTREAM["calls"] += 1
    if DOWNSTREAM["calls"] > DOWNSTREAM["capacity"]:
        DOWNSTREAM["rejected"] += 1
        return {"error": "capacity exceeded"}
    return {"result": None}                     # no match, ever

def agent_loop(task, max_steps=None):
    """plan -> act -> observe -> decide again. Stops when it succeeds."""
    steps, tokens = 0, 0
    while True:
        steps += 1
        tokens += 1800
        result = flaky_api(task)
        if result.get("result"):
            return {"done": True, "steps": steps, "tokens": tokens}
        if max_steps and steps >= max_steps:
            return {"done": False, "steps": steps, "tokens": tokens, "stopped_by": "budget"}
        if steps > 500:                          # the notebook's own safety net
            return {"done": False, "steps": steps, "tokens": tokens, "stopped_by": "runaway"}

r = agent_loop("find the order for customer 99999")     # this order does not exist
print(f"steps taken           : {r['steps']}")
print(f"tokens spent          : {r['tokens']:,}  (about ${r['tokens']/1000*0.002:,.2f})")
print(f"downstream calls      : {DOWNSTREAM['calls']}")
print(f"downstream rejections : {DOWNSTREAM['rejected']}  <- other callers got these")
print(f"stopped by            : {r['stopped_by']}")
print()
print("The agent was not attacked and nothing malfunctioned. It was given a")
print("task that cannot succeed, and the loop did what loops do.")
print()
print(f"{DOWNSTREAM['rejected']} rejections went to whoever else was using that")
print("service - a denial of service launched from inside the perimeter, by")
print("something holding valid credentials.")
assert DOWNSTREAM["rejected"] > 0

## What you just proved

An agent given an impossible task loops until the notebook's own safety net stops it, spending hundreds of thousands of tokens and exhausting a downstream service's capacity — with the rejections landing on whoever else was using that service.

## Your turn

Find the ceiling on one agent loop you run. If there is a token budget but no cap on downstream calls, the cost is bounded and the availability risk is not.

---

**Next → [A1.13 · Repudiation and untraceability](https://spbreed.github.io/cyber-commons/lessons/A1.13.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.12.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.12.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*